In [0]:
from pyspark.sql import Row
from datetime import datetime
results = []
def check(name, passed, detail=""):
    results.append(Row(check_name=name, status="PASS" if passed else "FAIL",
                       detail=str(detail), run_ts=datetime.now()))

# 1. Row count: bronze vs silver (allowing quarantined rows)
b = spark.table("workspace.bronze.raw_orders").select("order_id").distinct().count()
s = spark.table("workspace.silver.clean_orders").count()
check("rowcount_orders_bronze_vs_silver", b == s, f"{b} vs {s}")
# 2. PK uniqueness
d = spark.sql("SELECT count(*) - count(DISTINCT sale_id) FROM workspace.silver.clean_pos_sales").first()[0]
check("pk_unique_pos_sales", d == 0, d)
# 3. Not-null keys
n = spark.table("workspace.silver.clean_orders").filter("cust_id IS NULL").count()
check("notnull_orders_cust_id", n == 0, n)
# 4. Sum reconciliation
sb = spark.sql("SELECT round(sum(order_amt),2) FROM workspace.bronze.raw_orders").first()[0]
ss = spark.sql("SELECT round(sum(order_amt),2) FROM workspace.silver.clean_orders").first()[0]
check("sum_orders_amt", sb == ss, f"{sb} vs {ss}")

spark.createDataFrame(results).write.mode("append").saveAsTable("workspace.dq.dq_results")
display(spark.table("workspace.dq.dq_results"))

check_name,status,detail,run_ts
rowcount_orders_bronze_vs_silver,PASS,20000 vs 20000,2026-11-07T14:27:32.911Z
pk_unique_pos_sales,PASS,0,2026-11-07T14:27:33.666Z
notnull_orders_cust_id,PASS,0,2026-11-07T14:27:34.241Z
sum_orders_amt,PASS,9112404.81 vs 9112404.81,2026-11-07T14:27:35.712Z
